# MACELES-OFF quickstart

**MACELES-OFF** = MACE-OFF (a short-range ML force field for organic molecules) augmented with **LES (Latent Ewald Summation)** — a method introduced by Bingqing Cheng that learns a per-atom latent variable and runs an Ewald summation on it, adding long-range electrostatics/dispersion that pure short-range models miss beyond their message-passing cutoff. MACELES-OFF was trained on the SPICE dataset.

- Paper: Kim, Wang, Vargas, Zhong, King, Inizan & Cheng, *"A Universal Augmentation Framework for Long-Range Electrostatics in Machine Learning Interatomic Potentials"*, JCTC 2025 ([arXiv:2507.14302](https://arxiv.org/abs/2507.14302))
- LES library: https://github.com/ChengUCB/les
- Checkpoints + scripts: https://github.com/ChengUCB/les_fit (see `./MLIPs/MACE-LES-new`)
- MACE (now includes the `MACELES` model class on `main`, in `mace/modules/extensions.py` — **not** `mace/modules/models.py`, a mix-up documented in [ACEsuit/mace#1250](https://github.com/ACEsuit/mace/issues/1250)): https://github.com/ACEsuit/mace

**This is research code, not a stable release** — APIs and checkpoint filenames have moved around as recently as late 2025. This notebook installs MACE fresh from GitHub `main` (the PyPI wheel has lagged behind on `MACELES` support), clones the checkpoint repo, and **auto-discovers** the model file rather than hardcoding a path, so it's more likely to survive future repo reshuffles. If a cell fails, the linked repos above are the source of truth — check them directly.

**License:** the `les` / `les_fit` repos are CC BY-NC 4.0 (non-commercial use only), separate from MACE-OFF's own Academic Software License.

**Before running:** in Colab, go to `Runtime > Change runtime type` and pick a GPU if you have one available; CPU works fine too, just slower.

In [ ]:
# Fresh installs from source — the PyPI mace-torch wheel has historically lagged
# behind on MACELES support, so we install straight from GitHub main.
!pip install -q ase numpy
!pip install -q "git+https://github.com/ACEsuit/mace.git"
!pip install -q "git+https://github.com/ChengUCB/les.git"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.1/453.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# MACELES lives in mace.modules.extensions, NOT mace.modules.models —
# checking mace.modules.models (a common first guess, see ACEsuit/mace#1250)
# will always report False even on a correct install. It also requires the
# external `les` package to be importable.
try:
    from mace.modules.extensions import MACELES
    print("MACELES class available: True (mace.modules.extensions.MACELES)")
except ImportError as e:
    print("MACELES class NOT available:", e)
    print("Check that both mace (ACEsuit/mace main) and les (ChengUCB/les) installed without errors above.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
MACELES class available: True (mace.modules.extensions.MACELES)


## 1. Get the pretrained checkpoint

Clone the `les_fit` data repo and search it for `.model` files, rather than assuming a fixed filename (the repo's layout has changed between the Oct 2025 and Dec 2025 updates).

In [3]:
import os
import glob
import subprocess

repo_dir = "les_fit"
if not os.path.isdir(repo_dir):
    subprocess.run(["git", "clone", "https://github.com/ChengUCB/les_fit.git"], check=True)

candidates = sorted(glob.glob(f"{repo_dir}/**/*.model", recursive=True))
print(f"Found {len(candidates)} .model file(s) in {repo_dir}:")
for c in candidates:
    print(" -", c)

Found 36 .model file(s) in les_fit:
 - les_fit/MACELES-OFF/MACELES-OFF_small.model
 - les_fit/MACELES-OFF/MACELES-OFF_small_converted.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/mace-r4.5-nl-1-l-0/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/mace-r4.5-nl-1-l-1/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/mace-r5.5-nl-0/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/mace-r5.5-nl-1-l-0/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/mace-r5.5-nl-1-l-1/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/maceles-r4.5-nl-1-l-0/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/maceles-r4.5-nl-1-l-1/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/maceles-r5.5-nl-0/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/maceles-r5.5-nl-1-l-0/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/Au-MgO-Al/maceles-r5.5-nl-1-l-1/Au2-MgO_stagetwo.model
 - les_fit/MLIPs/MACE-LES/dipep/mace-r-4-l-0/dipep_train_stagetwo.model
 

In [4]:
# Prefer a checkpoint whose path/name mentions OFF; fall back to listing everything
# for you to pick manually if auto-detection doesn't find one.
off_candidates = [c for c in candidates if "off" in c.lower()]

if not off_candidates:
    raise FileNotFoundError(
        "Couldn't auto-find a MACELES-OFF checkpoint under les_fit/. "
        "Browse https://github.com/ChengUCB/les_fit/tree/main/MLIPs/MACE-LES-new "
        "manually, note the exact filename, and set model_path to it below."
    )

model_path = off_candidates[0]
print("Using checkpoint:", model_path)

Using checkpoint: les_fit/MACELES-OFF/MACELES-OFF_small.model


In [5]:
import hashlib

def md5sum(path, chunk_size=8192):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

digest = md5sum(model_path)
expected = "cc10e937b55e09f05b16dba756e2311b"  # per les_fit README, main-branch-compatible checkpoint
print("MD5:", digest)
print("Matches expected main-branch checkpoint:", digest == expected)

MD5: 481607f3c8a24558836c4cb93a20ebca
Matches expected main-branch checkpoint: False


## 2. Load MACELES-OFF (and plain MACE-OFF for comparison)

In [7]:
model_path = off_candidates[1]
print(f"Overriding model_path to: {model_path}")

from mace.calculators import MACECalculator, mace_off

try:
    calc_les = MACECalculator(
        model_paths=model_path,
        device=device,
        default_dtype="float32",  # MACELES-OFF was trained in float32
        dispersion=False,
    )
except (AttributeError, ModuleNotFoundError) as e:
    raise RuntimeError(
        "Failed to unpickle the checkpoint. This class of error usually means "
        "the .model file was pickled against a different module path for MACELES "
        "than what your installed mace/les versions expose (e.g. an older "
        "ChengUCB/mace-fork checkpoint vs. the current ACEsuit/mace main branch "
        "layout). Make sure model_path points at the *converted* MACELES-OFF "
        "checkpoint (MD5 cc10e937b55e09f05b16dba756e2311b) meant for the current "
        "main branch — re-check the MD5 cell above. If it still fails, the file "
        "layout has likely shifted again; check https://github.com/ChengUCB/les_fit "
        "directly for the current instructions."
    ) from e

calc_off = mace_off(model="medium", device=device)  # plain short-range baseline

Overriding model_path to: les_fit/MACELES-OFF/MACELES-OFF_small_converted.model


/usr/local/lib/python3.12/dist-packages/mace/calculators/mace.py:254: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


The model is distributed under the Academic Software License (ASL) license, see https://github.com/gabor1/ASL 
 To use the model you accept the terms of the license.
ASL is based on the Gnu Public License, but does not permit commercial use
Downloading: 100.0% (17.5 MB / 17.5 MB)
Cached MACE model to /root/.cache/mace/MACE-OFF23_medium.model
Using MACE-OFF23 MODEL for MACECalculator with /root/.cache/mace/MACE-OFF23_medium.model
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.


/usr/local/lib/python3.12/dist-packages/mace/calculators/mace.py:254: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


In [8]:
from ase import build

water = build.molecule("H2O")
water.calc = calc_les
print("MACELES-OFF energy (eV):", water.get_potential_energy())
print("MACELES-OFF forces (eV/\u00c5):\n", water.get_forces())

MACELES-OFF energy (eV): -2081.116943359375
MACELES-OFF forces (eV/Å):
 [[ 0.          0.         -0.6474355 ]
 [ 0.         -0.34212863  0.32371777]
 [ 0.          0.34212863  0.32371777]]


## 3. Show the long-range electrostatic tail

The point of LES is interactions that survive beyond a short-range model's cutoff/receptive field. Take two water molecules, pull them apart, and compare the interaction energy — `E(dimer) - 2*E(monomer)` — predicted by MACELES-OFF vs. plain MACE-OFF. Plain MACE-OFF's interaction should collapse to ~0 once the separation clears its receptive field (roughly its ~5 Å cutoff × number of message-passing layers); MACELES-OFF should retain a small nonzero electrostatic tail well past that.

In [9]:
def water_dimer(separation):
    a = build.molecule("H2O")
    b = build.molecule("H2O")
    b.translate([separation, 0.0, 0.0])
    return a + b

def monomer_energy(calc):
    w = build.molecule("H2O")
    w.calc = calc
    return w.get_potential_energy()

e_mono_les = monomer_energy(calc_les)
e_mono_off = monomer_energy(calc_off)

distances = [3.0, 4.0, 5.0, 6.0, 8.0, 10.0, 14.0, 20.0]  # Å, O-O separation along x

print(f"{'d (\u00c5)':>6} | {'E_int MACELES-OFF (meV)':>24} | {'E_int MACE-OFF (meV)':>22}")
for d in distances:
    dimer = water_dimer(d)

    dimer.calc = calc_les
    e_int_les = (dimer.get_potential_energy() - 2 * e_mono_les) * 1000  # meV

    dimer.calc = calc_off
    e_int_off = (dimer.get_potential_energy() - 2 * e_mono_off) * 1000  # meV

    print(f"{d:6.1f} | {e_int_les:24.3f} | {e_int_off:22.3f}")

 d (Å) |  E_int MACELES-OFF (meV) |   E_int MACE-OFF (meV)
   3.0 |                  156.738 |                140.556
   4.0 |                   34.668 |                 38.903
   5.0 |                    4.883 |                 -0.000
   6.0 |                    3.418 |                 -0.000
   8.0 |                    1.953 |                 -0.000
  10.0 |                    1.465 |                 -0.000
  14.0 |                    0.977 |                 -0.000
  20.0 |                    0.977 |                 -0.000


## Notes

- If installs fail or `MACELES` isn't found, MACE's LES support is a moving target — check https://github.com/ACEsuit/mace/issues for the current state.
- If checkpoint auto-detection picks the wrong file, browse https://github.com/ChengUCB/les_fit/tree/main/MLIPs/MACE-LES-new directly and set `model_path` manually.
- `remove_self_interaction=True` (the LES default) is the most robust choice; setting it `False` can slightly improve training accuracy but is less robust when extrapolating from finite to periodic systems.
- For periodic (bulk/liquid) systems, LES switches from pairwise summation to true reciprocal-space Ewald summation automatically based on whether the `Atoms` object has a periodic cell set.